# Pneumonia Detection from Chest X-rays: 3 CNNs + Score-CAM

**Models:** ResNet-50, DenseNet-121, EfficientNet-B0 (ImageNet pretrained, fully fine-tuned)
**Explainability:** Score-CAM implemented from scratch (gradient-free, no library)
**Data:** Kaggle *Chest X-Ray Images (Pneumonia)*, patient-aware train/val split
**Evaluation:** temperature calibration, validation-chosen thresholds, patient-level bootstrap CIs, McNemar tests, shortcut check

Run the cells top to bottom. Everything (checkpoints, metrics, figures) is saved to Google Drive so you can reuse it for the Streamlit version.

> Research/education only. Not a medical device.

## 1. Setup

In [ ]:
!nvidia-smi -L
!pip -q install kagglehub scikit-learn seaborn tqdm

In [ ]:
import os, glob, json, random, time, copy, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

warnings.filterwarnings("ignore")
SEED = 42
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("Device:", device, "| torch", torch.__version__)

## 2. Config and Google Drive output folder

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CFG = dict(
    img_size=224,
    batch_size=32,
    epochs=12,
    lr=1e-4,
    weight_decay=1e-4,
    patience=4,            # early stopping on validation loss
    val_folds=7,           # 1 of 7 folds (~14%) becomes validation
    num_workers=2,
)
FORCE_RETRAIN = False       # True = ignore saved checkpoints and retrain
MODEL_NAMES = ["resnet50", "densenet121", "efficientnet_b0"]
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

OUT_DIR = Path("/content/drive/MyDrive/pneumonia_project")
MODEL_DIR, FIG_DIR, RES_DIR = OUT_DIR/"models", OUT_DIR/"figures", OUT_DIR/"results"
for d in (MODEL_DIR, FIG_DIR, RES_DIR): d.mkdir(parents=True, exist_ok=True)
print("Outputs ->", OUT_DIR)

## 3. Load the dataset directly from Kaggle
Uses `kagglehub`, which downloads the public dataset straight from Kaggle. No `kaggle.json` or manual upload is needed.

In [ ]:
import kagglehub
DATA_PATH = Path(kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"))
print("Downloaded to:", DATA_PATH)

In [ ]:
cands = [p for p in DATA_PATH.rglob("train")
         if p.is_dir() and (p.parent/"test").is_dir() and "__MACOSX" not in str(p)]
ROOT = min(cands, key=lambda p: len(p.parts)).parent     # shallowest match
print("Dataset root:", ROOT)

def collect(split_dir):
    rows = []
    for cls, lab in CLASS_TO_IDX.items():
        for p in sorted((split_dir/cls).glob("*")):
            if p.suffix.lower() in {".jpeg", ".jpg", ".png"} and not p.name.startswith("._"):
                rows.append((str(p), lab))
    return pd.DataFrame(rows, columns=["path", "label"])

orig_train, orig_val, test_df = collect(ROOT/"train"), collect(ROOT/"val"), collect(ROOT/"test")
trainval = pd.concat([orig_train, orig_val], ignore_index=True)
print("train+val:", len(trainval), "| test:", len(test_df))

## 4. Patient-aware split

The original `val` folder has only 16 images, which is too small to trust. The same patient also appears in several images, so a plain random split would **leak** patients between train and validation. We rebuild train/val with `StratifiedGroupKFold`, grouping by a patient ID parsed from the file name (best-effort), and **verify no group overlap**. The official `test` folder is untouched.

In [ ]:
def patient_id(path, label):
    stem = Path(path).stem
    if label == 1:                    # person123_bacteria_456 -> person123
        return stem.split("_")[0]
    return stem.rsplit("-", 1)[0]     # IM-0115-0001 -> IM-0115 ; NORMAL2-IM-0381-0001 -> NORMAL2-IM-0381

trainval["group"] = [patient_id(p, l) for p, l in zip(trainval.path, trainval.label)]
sgkf = StratifiedGroupKFold(n_splits=CFG["val_folds"], shuffle=True, random_state=SEED)
tr_idx, va_idx = next(iter(sgkf.split(trainval, trainval.label, groups=trainval.group)))
train_df, val_df = trainval.iloc[tr_idx].reset_index(drop=True), trainval.iloc[va_idx].reset_index(drop=True)

overlap = set(train_df.group) & set(val_df.group)
print("Patient-group overlap train/val:", len(overlap), "(must be 0)")
assert len(overlap) == 0

summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "NORMAL":    [(d.label == 0).sum() for d in (train_df, val_df, test_df)],
    "PNEUMONIA": [(d.label == 1).sum() for d in (train_df, val_df, test_df)],
})
summary["total"] = summary.NORMAL + summary.PNEUMONIA
display(summary)
summary.to_csv(RES_DIR/"split_summary.csv", index=False)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for r, lab in enumerate([0, 1]):
    for c, p in enumerate(train_df[train_df.label == lab].sample(4, random_state=SEED).path):
        axes[r, c].imshow(Image.open(p).convert("L"), cmap="gray"); axes[r, c].axis("off")
        axes[r, c].set_title(CLASS_NAMES[lab])
plt.tight_layout(); plt.show()

## 5. Datasets, transforms, class weights

In [ ]:
S = CFG["img_size"]
train_tf = transforms.Compose([
    transforms.Resize((S, S)),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize((S, S)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class XrayDataset(Dataset):
    def __init__(self, df, tf): self.paths, self.labels, self.tf = df.path.tolist(), df.label.tolist(), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"], pin_memory=True)
train_loader = DataLoader(XrayDataset(train_df, train_tf), shuffle=True, drop_last=True, **kw)
val_loader   = DataLoader(XrayDataset(val_df, eval_tf), shuffle=False, **kw)
test_loader  = DataLoader(XrayDataset(test_df, eval_tf), shuffle=False, **kw)

counts = train_df.label.value_counts().sort_index().values
class_weights = torch.tensor(len(train_df) / (2.0 * counts), dtype=torch.float32)
print("Class counts:", counts, "-> loss weights:", class_weights.numpy().round(3))

## 6. Models
All three are fully fine-tuned. Score-CAM needs the **last convolutional feature map** of each, defined in `get_target_layer`.

In [ ]:
def build_model(name):
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 2))
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier.in_features, 2))
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 2))
    else:
        raise ValueError(name)
    return m

TARGET_LAYER_EXPR = {                      # kept as strings so the Streamlit app can reuse them
    "resnet50": "model.layer4[-1]",
    "densenet121": "model.features.norm5",
    "efficientnet_b0": "model.features[-1]",
}
def get_target_layer(model, name):
    return eval(TARGET_LAYER_EXPR[name], {"model": model})

for n in MODEL_NAMES:
    m = build_model(n)
    print(f"{n:16s} params: {sum(p.numel() for p in m.parameters())/1e6:5.1f}M | Score-CAM layer: {TARGET_LAYER_EXPR[n]}")
    del m

## 7. Training loop
AdamW + cosine schedule, mixed precision, class-weighted cross-entropy, early stopping on validation loss. The best checkpoint (lowest val loss) is kept.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None, desc=""):
    training = optimizer is not None
    model.train(training)
    total_loss, n, probs, labels = 0.0, 0, [], []
    for x, y in tqdm(loader, desc=desc, leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * x.size(0); n += x.size(0)
        probs.append(torch.softmax(logits.float(), 1)[:, 1].detach().cpu()); labels.append(y.cpu())
    probs, labels = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    acc = accuracy_score(labels, probs >= 0.5)
    auc = roc_auc_score(labels, probs)
    return total_loss / n, acc, auc

def train_model(name):
    ckpt, hist_path = MODEL_DIR/f"{name}.pt", RES_DIR/f"history_{name}.csv"
    model = build_model(name).to(device)
    if ckpt.exists() and hist_path.exists() and not FORCE_RETRAIN:
        print(f"[{name}] checkpoint found -> loading (set FORCE_RETRAIN=True to retrain)")
        model.load_state_dict(torch.load(ckpt, map_location=device))
        return model.eval(), pd.read_csv(hist_path)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

    best, bad, rows = float("inf"), 0, []
    for ep in range(1, CFG["epochs"] + 1):
        t0 = time.time()
        tl, ta, tauc = run_epoch(model, train_loader, criterion, optimizer, scaler, f"{name} ep{ep} train")
        vl, va, vauc = run_epoch(model, val_loader, criterion, desc=f"{name} ep{ep} val")
        scheduler.step()
        rows.append(dict(epoch=ep, train_loss=tl, train_acc=ta, train_auc=tauc,
                         val_loss=vl, val_acc=va, val_auc=vauc))
        print(f"[{name}] ep{ep:02d} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f} auc {vauc:.4f} | {time.time()-t0:.0f}s")
        if vl < best:
            best, bad = vl, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= CFG["patience"]:
                print(f"[{name}] early stopping at epoch {ep}"); break

    model.load_state_dict(torch.load(ckpt, map_location=device))
    history = pd.DataFrame(rows); history.to_csv(hist_path, index=False)
    return model.eval(), history

In [ ]:
trained, histories = {}, {}
for name in MODEL_NAMES:
    trained[name], histories[name] = train_model(name)

In [ ]:
fig, axes = plt.subplots(len(MODEL_NAMES), 3, figsize=(15, 4 * len(MODEL_NAMES)))
for r, n in enumerate(MODEL_NAMES):
    h = histories[n]
    for c, (metric, title) in enumerate([("loss", "Loss"), ("acc", "Accuracy"), ("auc", "AUC")]):
        axes[r, c].plot(h.epoch, h[f"train_{metric}"], label="train")
        axes[r, c].plot(h.epoch, h[f"val_{metric}"], label="val")
        axes[r, c].set_title(f"{n} - {title}"); axes[r, c].set_xlabel("epoch"); axes[r, c].legend()
plt.tight_layout(); plt.savefig(FIG_DIR/"training_curves.png", dpi=150); plt.show()

## 8. Logits on validation and test, and probability calibration

Fine-tuned CNNs are usually **over-confident** (many outputs of exactly 1.00). We fit one **temperature** `T` per model on the *validation* set (`softmax(logits / T)`). This is monotone, so it does not change a single model's AUC or ranking, but it makes the probabilities more honest. Quality is measured with ECE (expected calibration error) and the Brier score. The ensemble averages the calibrated probabilities.

In [ ]:
@torch.no_grad()
def predict_logits(model, loader):
    model.eval(); out, labels = [], []
    for x, y in tqdm(loader, desc="predict", leave=False):
        out.append(model(x.to(device)).float().cpu()); labels.append(y)
    return torch.cat(out).numpy(), torch.cat(labels).numpy()

val_logits, test_logits = {}, {}
for n in MODEL_NAMES:
    val_logits[n], y_val = predict_logits(trained[n], val_loader)
    test_logits[n], y_test = predict_logits(trained[n], test_loader)
assert (y_test == test_df.label.values).all() and (y_val == val_df.label.values).all()

def softmax_p1(logits, T=1.0):
    z = logits / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e[:, 1] / e.sum(axis=1)

def fit_temperature(logits, labels):
    lg = torch.tensor(logits, dtype=torch.float32); y = torch.tensor(labels, dtype=torch.long)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=300, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(lg / log_t.exp(), y); loss.backward(); return loss
    opt.step(closure)
    return float(log_t.exp().item())

TEMP = {n: fit_temperature(val_logits[n], y_val) for n in MODEL_NAMES}
print("Fitted temperatures:", {k: round(v, 3) for k, v in TEMP.items()})

val_p_raw  = {n: softmax_p1(val_logits[n]) for n in MODEL_NAMES}
val_p_cal  = {n: softmax_p1(val_logits[n], TEMP[n]) for n in MODEL_NAMES}
test_p_raw = {n: softmax_p1(test_logits[n]) for n in MODEL_NAMES}
test_p_cal = {n: softmax_p1(test_logits[n], TEMP[n]) for n in MODEL_NAMES}
for D in (val_p_raw, val_p_cal, test_p_raw, test_p_cal):
    D["ensemble"] = np.mean([D[n] for n in MODEL_NAMES], axis=0)
ALL = MODEL_NAMES + ["ensemble"]

def ece_score(y, p1, bins=10):
    conf = np.maximum(p1, 1 - p1); correct = ((p1 >= 0.5).astype(int) == y).astype(float)
    edges = np.linspace(0.5, 1.0, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return e

cal_table = pd.DataFrame([dict(
    model=n, temperature=TEMP.get(n, np.nan),
    ece_val_raw=ece_score(y_val, val_p_raw[n]), ece_val_cal=ece_score(y_val, val_p_cal[n]),
    ece_test_raw=ece_score(y_test, test_p_raw[n]), ece_test_cal=ece_score(y_test, test_p_cal[n]),
    brier_test_raw=np.mean((test_p_raw[n] - y_test) ** 2), brier_test_cal=np.mean((test_p_cal[n] - y_test) ** 2),
) for n in ALL]).set_index("model")
cal_table.to_csv(RES_DIR/"calibration.csv")
display(cal_table.round(4))

## 9. Operating thresholds (chosen on validation only)

Three decision rules are compared. **The test set is used once, after the thresholds are fixed.**

- `default`: 0.5
- `youden`: threshold maximising sensitivity + specificity on validation
- `sens98`: highest threshold that still reaches 98% sensitivity on validation (a screening-style operating point)

Caveat: validation was also used for checkpoint selection and temperature fitting, so validation numbers are optimistic. If test data is shifted relative to validation (it is here), a validation-chosen threshold may not transfer.

In [ ]:
def pick_threshold(y, p, mode, target_sens=0.98):
    fpr, tpr, thr = roc_curve(y, p)
    if mode == "youden":   i = int(np.argmax(tpr - fpr))
    elif mode == "sens98": i = int(np.where(tpr >= target_sens)[0][0])
    else: raise ValueError(mode)
    return float(min(max(thr[i], 0.0), 1.0))

THRESH = {"default": {n: 0.5 for n in ALL}}
for mode in ("youden", "sens98"):
    THRESH[mode] = {n: pick_threshold(y_val, val_p_cal[n], mode) for n in ALL}
display(pd.DataFrame(THRESH).round(4))

def confusion_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(accuracy=(tp + tn) / len(y), sensitivity=tp / max(tp + fn, 1), specificity=tn / max(tn + fp, 1),
                precision=tp / max(tp + fp, 1), f1=2 * tp / max(2 * tp + fp + fn, 1), TP=tp, FP=fp, TN=tn, FN=fn)

print("Validation sensitivity / specificity at the chosen thresholds:")
vrows = []
for mode in ("youden", "sens98"):
    for n in ALL:
        m = confusion_metrics(y_val, val_p_cal[n], THRESH[mode][n])
        vrows.append(dict(rule=mode, model=n, threshold=THRESH[mode][n], sensitivity=m["sensitivity"], specificity=m["specificity"]))
display(pd.DataFrame(vrows).round(4))

## 10. Test-set results
Uncalibrated @ 0.5 is what you had before; the others use calibrated probabilities and a validation-chosen threshold.

In [ ]:
rows = []
for n in ALL:
    ops = {"raw@0.5": (test_p_raw, 0.5), "cal@0.5": (test_p_cal, 0.5),
           "cal@youden(val)": (test_p_cal, THRESH["youden"][n]), "cal@sens98(val)": (test_p_cal, THRESH["sens98"][n])}
    for op, (P, thr) in ops.items():
        rows.append(dict(model=n, operating_point=op, threshold=round(thr, 4),
                         auc=roc_auc_score(y_test, P[n]), **confusion_metrics(y_test, P[n], thr)))
op_table = pd.DataFrame(rows)
op_table.to_csv(RES_DIR/"test_metrics_all_operating_points.csv", index=False)
display(op_table.round(4))

def spec_at_sens(y, p, s):
    fpr, tpr, _ = roc_curve(y, p); i = np.where(tpr >= s)[0][0]; return 1 - fpr[i]
rank_table = pd.DataFrame({n: {"auc": roc_auc_score(y_test, test_p_cal[n]),
                               "specificity@95%sens": spec_at_sens(y_test, test_p_cal[n], 0.95),
                               "specificity@98%sens": spec_at_sens(y_test, test_p_cal[n], 0.98)} for n in ALL}).T
print("Ranking quality read off the ROC curve (descriptive, not an operating point):")
display(rank_table.round(4))
rank_table.to_csv(RES_DIR/"test_ranking_quality.csv")

In [ ]:
OPERATING_POINT = "youden"      # "default" | "youden" | "sens98"  (used for figures, CIs and the app config)
final_thr = THRESH[OPERATING_POINT]
print("Operating point:", OPERATING_POINT, {k: round(v, 3) for k, v in final_thr.items()})

## 11. Uncertainty: patient-level bootstrap, paired AUC differences, McNemar

The test set is small (624 images) and several images can come from one patient, so images are **not independent**. The bootstrap resamples **patient groups** (parsed from file names, best-effort) 2000 times. The same resamples are used for all models, so differences between models are paired. McNemar's exact test compares decisions on the same images (it ignores patient clustering, so treat p-values as indicative).

In [ ]:
import itertools
from scipy.stats import binomtest

test_groups = np.array([patient_id(p, l) for p, l in zip(test_df.path, test_df.label)])
uniq = np.unique(test_groups)
group_idx = [np.where(test_groups == g)[0] for g in uniq]
print(f"{len(test_df)} test images from {len(uniq)} patient groups")

def cluster_bootstrap(y, probs, thrs, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    res = {n: {k: [] for k in ("accuracy", "sensitivity", "specificity", "auc")} for n in probs}
    for _ in range(n_boot):
        pick = rng.integers(0, len(group_idx), len(group_idx))
        idx = np.concatenate([group_idx[g] for g in pick])
        yb = y[idx]
        if yb.min() == yb.max(): continue
        for n in probs:
            pb = probs[n][idx]; pred = pb >= thrs[n]
            res[n]["accuracy"].append((pred == yb).mean())
            res[n]["sensitivity"].append(pred[yb == 1].mean())
            res[n]["specificity"].append((~pred[yb == 0]).mean())
            res[n]["auc"].append(roc_auc_score(yb, pb))
    return {n: {k: np.array(v) for k, v in d.items()} for n, d in res.items()}

boot = cluster_bootstrap(y_test, test_p_cal, final_thr)
def ci(a): return np.percentile(a, [2.5, 97.5])

rows = []
for n in ALL:
    m = confusion_metrics(y_test, test_p_cal[n], final_thr[n]); m["auc"] = roc_auc_score(y_test, test_p_cal[n])
    row = {"model": n, "threshold": round(final_thr[n], 3)}
    for k in ("auc", "accuracy", "sensitivity", "specificity"):
        lo, hi = ci(boot[n][k]); row[k] = f"{m[k]:.3f} [{lo:.3f}, {hi:.3f}]"
    rows.append(row)
ci_table = pd.DataFrame(rows).set_index("model")
ci_table.to_csv(RES_DIR/"test_metrics_with_CI.csv")
print(f"Test metrics at operating point '{OPERATING_POINT}' with 95% patient-level bootstrap CIs:")
display(ci_table)

prow = []
for a, b in itertools.combinations(ALL, 2):
    d = boot[a]["auc"] - boot[b]["auc"]; lo, hi = ci(d)
    prow.append(dict(pair=f"{a} - {b}", mean_auc_diff=d.mean(), ci_low=lo, ci_high=hi, excludes_zero=bool(lo > 0 or hi < 0)))
print("Paired AUC differences:")
display(pd.DataFrame(prow).round(4))

def mcnemar_exact(c1, c2):
    b = int(np.sum(c1 & ~c2)); c = int(np.sum(~c1 & c2))
    p = 1.0 if b + c == 0 else binomtest(min(b, c), b + c, 0.5).pvalue
    return b, c, p
correct = {n: ((test_p_cal[n] >= final_thr[n]).astype(int) == y_test) for n in ALL}
mrow = []
pairs = list(itertools.combinations(ALL, 2))
for a, b in pairs:
    nb, nc, p = mcnemar_exact(correct[a], correct[b])
    mrow.append(dict(pair=f"{a} vs {b}", only_first_correct=nb, only_second_correct=nc, p_value=p, p_bonferroni=min(1.0, p * len(pairs))))
print("McNemar exact test on per-image correctness:")
display(pd.DataFrame(mrow).round(4))

## 12. Figures: ROC, reliability diagrams, confusion matrices

In [ ]:
from sklearn.calibration import calibration_curve

plt.figure(figsize=(6.5, 6))
for n in ALL:
    fpr, tpr, _ = roc_curve(y_test, test_p_cal[n])
    plt.plot(fpr, tpr, label=f"{n} (AUC {roc_auc_score(y_test, test_p_cal[n]):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC - test set"); plt.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/"roc_curves.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, len(ALL), figsize=(4.5 * len(ALL), 4.5))
for ax, n in zip(axes, ALL):
    for lab, P in (("raw", test_p_raw), ("calibrated", test_p_cal)):
        frac, mean = calibration_curve(y_test, P[n], n_bins=10, strategy="uniform")
        ax.plot(mean, frac, marker="o", label=lab)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_title(f"{n} reliability (test)"); ax.set_xlabel("Predicted P(pneumonia)"); ax.set_ylabel("Observed fraction"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/"reliability.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, len(ALL), figsize=(4.5 * len(ALL), 4))
for ax, n in zip(axes, ALL):
    cm = confusion_matrix(y_test, (test_p_cal[n] >= final_thr[n]).astype(int), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{n} @ {final_thr[n]:.2f}"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig(FIG_DIR/"confusion_matrices.png", dpi=150); plt.show()

## 13. Shortcut check: what happens if we hide parts of the image?

Test AUC with (a) only the **centre** visible (outer 15% frame zeroed) and (b) only the **frame** visible (centre 70% zeroed). If AUC with the frame alone is far above 0.5, the model can classify using regions that contain little or no lung (shoulders, markers, borders, positioning, scanner differences). This is an indicative check, not proof: the frame still includes the outer lung edges, and the models never saw such masked images in training.

In [ ]:
def region_mask(kind, size, margin=0.15):
    k = int(size * margin)
    if kind == "center_only":
        m = torch.zeros(1, 1, size, size); m[..., k:size - k, k:size - k] = 1
    elif kind == "frame_only":
        m = torch.ones(1, 1, size, size); m[..., k:size - k, k:size - k] = 0
    else:
        raise ValueError(kind)
    return m

@torch.no_grad()
def logits_with_mask(model, loader, mask):
    model.eval(); out = []; mask = mask.to(device)
    for x, _ in loader:
        out.append(model(x.to(device) * mask).float().cpu())     # 0 in normalised space = dataset-mean grey
    return torch.cat(out).numpy()

rows = []
for n in MODEL_NAMES:
    row = {"model": n, "auc_full_image": roc_auc_score(y_test, test_p_cal[n])}
    for kind in ("center_only", "frame_only"):
        lg = logits_with_mask(trained[n], test_loader, region_mask(kind, S))
        row[f"auc_{kind}"] = roc_auc_score(y_test, softmax_p1(lg, TEMP[n]))
    rows.append(row)
shortcut = pd.DataFrame(rows).set_index("model")
shortcut.to_csv(RES_DIR/"shortcut_check.csv")
display(shortcut.round(4))

## 14. Score-CAM from scratch

Algorithm (Wang et al., CVPR-W 2020), per image and target class *c*:

1. Forward pass; capture activations **A** (K channels) of the last conv layer.
2. For each channel *k*: ReLU, upsample to input size, min-max normalise to [0,1] → mask **M_k**.
3. Score each masked input: `s_k = softmax(f(x ⊙ M_k))[c]`; baseline `s_0 = softmax(f(0))[c]`.
4. Channel-wise Increase of Confidence: `w_k = s_k − s_0`.
5. `CAM = ReLU(Σ_k w_k · A_k)`, upsample, normalise to [0,1].

All K channels are used (no subsampling). `weight_mode`: `"cic"` (paper), `"softmax"` (softmax-normalised CIC), `"score"` (raw `s_k`, no baseline).

**Known behaviour:** if the zero-image baseline already gets a high score for the target class, almost every `w_k` is ≤ 0 and the map is **empty**. The result dict exposes `empty`, `base_score` and `frac_pos` so this can be measured instead of guessed.

In [ ]:
class ScoreCAM:
    def __init__(self, model, target_layer, batch_size=64, weight_mode="cic", use_amp=True):
        assert weight_mode in ("cic", "softmax", "score")
        self.model, self.bs, self.mode = model.eval(), batch_size, weight_mode
        self.use_amp = use_amp and device.type == "cuda"
        self._acts, self._capture = None, True
        self._handle = target_layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        if self._capture:
            self._acts = out.detach().clone().float()    # clone: the next op may be an in-place ReLU

    def remove(self): self._handle.remove()

    def _scores(self, x, target):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=self.use_amp):
            out = self.model(x)
        return torch.softmax(out.float(), dim=1)[:, target]

    @torch.no_grad()
    def __call__(self, x, target_class=None):
        H, W = x.shape[-2:]
        self._capture = True
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=self.use_amp):
            logits = self.model(x)
        logits = logits.float()
        probs = torch.softmax(logits, 1)[0]
        target = int(probs.argmax()) if target_class is None else int(target_class)
        acts = F.relu(self._acts[0])                                    # (K,h,w)

        self._capture = False
        try:
            maps = F.interpolate(acts[None], size=(H, W), mode="bilinear", align_corners=False)[0]   # (K,H,W)
            flat = maps.flatten(1)
            mn, mx = flat.min(1)[0][:, None, None], flat.max(1)[0][:, None, None]
            rng = mx - mn
            constant = (rng.squeeze(-1).squeeze(-1) < 1e-8)
            maps = (maps - mn) / (rng + 1e-8)
            maps[constant] = 0.0

            base = self._scores(torch.zeros_like(x), target)[0]
            scores = []
            for i in range(0, maps.shape[0], self.bs):
                scores.append(self._scores(x * maps[i:i + self.bs].unsqueeze(1), target))
            scores = torch.cat(scores)
        finally:
            self._capture = True

        cic = scores - base
        if self.mode == "score":     w = scores.clone()
        else:                        w = cic.clone()
        w[constant] = 0.0
        if self.mode == "softmax":   w = torch.softmax(w, dim=0)

        cam_lr = F.relu((w[:, None, None] * acts).sum(0))
        empty = bool(cam_lr.max() <= 1e-8)
        cam = F.interpolate(cam_lr[None, None], size=(H, W), mode="bilinear", align_corners=False)[0, 0]
        cam = torch.zeros_like(cam) if empty else (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        valid = ~constant
        frac_pos = float((cic[valid] > 0).float().mean()) if valid.any() else 0.0
        return dict(cam=cam.cpu().numpy().astype(np.float32), target=target, prob_target=float(probs[target]),
                    probs=probs.cpu().numpy(), logits=logits[0].cpu().numpy(), empty=empty,
                    base_score=float(base), frac_pos=frac_pos,
                    n_channels=int(maps.shape[0]), n_constant=int(constant.sum()))

def load_tensor(path): return eval_tf(Image.open(path).convert("RGB")).unsqueeze(0).to(device)

def denorm(x):
    m = torch.tensor(MEAN, device=x.device)[:, None, None]; s = torch.tensor(STD, device=x.device)[:, None, None]
    return (x[0] * s + m).clamp(0, 1).permute(1, 2, 0).cpu().numpy()

def overlay(img, cam, alpha=0.5):
    return np.clip((1 - alpha) * img + alpha * plt.cm.jet(cam)[..., :3], 0, 1)

In [ ]:
CAM_MODE = "cic"       # "cic" (paper) | "softmax" | "score"
explainers = {n: ScoreCAM(trained[n], get_target_layer(trained[n], n), batch_size=64, weight_mode=CAM_MODE) for n in MODEL_NAMES}

# Why were NORMAL-class maps blank? Measure it on 3 normal test images, for both target classes.
normal_paths = test_df[test_df.label == 0].sample(3, random_state=SEED).path.tolist()
diag = []
for n in MODEL_NAMES:
    for path in normal_paths:
        x = load_tensor(path)
        for tgt in (0, 1):
            o = explainers[n](x, target_class=tgt)
            diag.append(dict(model=n, target=CLASS_NAMES[tgt], baseline_score=o["base_score"],
                             pct_channels_positive_CIC=100 * o["frac_pos"], empty_map=float(o["empty"])))
print(f"Score-CAM mode: {CAM_MODE}. Diagnostic on 3 normal images (mean over images):")
display(pd.DataFrame(diag).groupby(["model", "target"]).mean(numeric_only=True).round(3))

## 15. Visual explanations
Samples are grouped by the **ensemble** decision at the chosen operating point. By default every map explains the **PNEUMONIA** class (`TARGET_CLASS = 1`), so maps are comparable across images and models; an empty map is labelled *no positive evidence*. Set `TARGET_CLASS = None` to explain each model's own predicted class.

In [ ]:
TARGET_CLASS = 1               # 1 = PNEUMONIA evidence | 0 = NORMAL | None = predicted class
N_PER_GROUP = {"TP": 3, "TN": 3, "FP": 3, "FN": 3}

preds_out = test_df[["path", "label"]].copy()
for n in ALL:
    preds_out[f"p_raw_{n}"] = test_p_raw[n]; preds_out[f"p_cal_{n}"] = test_p_cal[n]
    preds_out[f"pred_{n}"] = (test_p_cal[n] >= final_thr[n]).astype(int)
preds_out["group"] = test_groups
preds_out.to_csv(RES_DIR/"test_predictions.csv", index=False)

pe = preds_out["pred_ensemble"]
groups = {"TP": preds_out[(preds_out.label == 1) & (pe == 1)], "TN": preds_out[(preds_out.label == 0) & (pe == 0)],
          "FP": preds_out[(preds_out.label == 0) & (pe == 1)], "FN": preds_out[(preds_out.label == 1) & (pe == 0)]}
print({k: len(v) for k, v in groups.items()})

def cam_grid(rows, title, fname):
    if len(rows) == 0:
        print(f"{title}: no samples"); return
    fig, axes = plt.subplots(len(rows), 1 + len(MODEL_NAMES), figsize=(4 * (1 + len(MODEL_NAMES)), 4 * len(rows)), squeeze=False)
    for r, (_, row) in enumerate(rows.iterrows()):
        x = load_tensor(row.path); img = denorm(x)
        axes[r, 0].imshow(img); axes[r, 0].axis("off")
        axes[r, 0].set_title(f"True: {CLASS_NAMES[int(row.label)]}\nEnsemble P(pneu)={row.p_cal_ensemble:.2f}", fontsize=10)
        for c, n in enumerate(MODEL_NAMES, start=1):
            out = explainers[n](x, target_class=TARGET_CLASS)
            p_cal = softmax_p1(out["logits"][None, :], TEMP[n])[0]
            axes[r, c].imshow(overlay(img, out["cam"])); axes[r, c].axis("off")
            note = "\n(no positive evidence)" if out["empty"] else ""
            axes[r, c].set_title(f"{n}\nexplains {CLASS_NAMES[out['target']]} | P(pneu)={p_cal:.2f}{note}", fontsize=10)
    fig.suptitle(title, fontsize=14, y=1.0)
    plt.tight_layout(); plt.savefig(FIG_DIR/fname, dpi=130, bbox_inches="tight"); plt.show()

titles = {"TP": "True positives (pneumonia correctly detected)", "TN": "True negatives (normal correctly detected)",
          "FP": "False positives (normal predicted as pneumonia)", "FN": "False negatives (pneumonia missed)"}
for g, k in N_PER_GROUP.items():
    sel = groups[g].sample(min(k, len(groups[g])), random_state=SEED) if len(groups[g]) else groups[g]
    cam_grid(sel, titles[g], f"scorecam_{g}.png")

In [ ]:
# High-confidence ensemble errors: candidates for label noise / ambiguous cases. Worth showing to a clinician.
hc = preds_out[(preds_out.label != preds_out.pred_ensemble) &
               ((preds_out.p_cal_ensemble > 0.95) | (preds_out.p_cal_ensemble < 0.05))]
hc.to_csv(RES_DIR/"review_high_confidence_errors.csv", index=False)
print(f"High-confidence ensemble errors (P>0.95 or P<0.05): {len(hc)} of {int((preds_out.label != preds_out.pred_ensemble).sum())} total errors")
display(hc[["path", "label", "p_cal_ensemble"]].head(10))

## 16. Quantitative check of the explanations

For each image, the Score-CAM map multiplies the input and the predicted class is re-scored (`Y` = original confidence, `O` = masked confidence).

- **Average Drop** (lower is better): mean of `max(0, Y − O) / Y × 100`
- **Increase in Confidence** (higher is better): % of images where `O > Y`
- **Empty maps**: % of images where Score-CAM found no positive evidence (the map is blank)

Results are split by the **predicted class**, because maps for NORMAL predictions behave differently (see the diagnostic above). Uses a class-balanced random test subset, so the group sizes depend on each model's predictions.

In [ ]:
N_PER_CLASS_EVAL = 25
eval_df = pd.concat([test_df[test_df.label == l].sample(N_PER_CLASS_EVAL, random_state=SEED) for l in (0, 1)])

records = []
for n in MODEL_NAMES:
    for path in tqdm(eval_df.path, desc=f"faithfulness {n}"):
        x = load_tensor(path)
        t0 = time.time(); out = explainers[n](x, target_class=None); dt = time.time() - t0
        cam_t = torch.from_numpy(out["cam"]).to(device)[None, None]
        with torch.no_grad():
            Y = float(torch.softmax(trained[n](x).float(), 1)[0, out["target"]])
            O = float(torch.softmax(trained[n](x * cam_t).float(), 1)[0, out["target"]])
        records.append(dict(model=n, predicted=CLASS_NAMES[out["target"]], drop=max(0.0, Y - O) / max(Y, 1e-8) * 100,
                            inc=O > Y, empty=out["empty"], sec=dt))
rec = pd.DataFrame(records)
faith = rec.groupby(["model", "predicted"]).agg(
    n_images=("drop", "size"), average_drop_pct=("drop", "mean"),
    increase_in_confidence_pct=("inc", lambda s: 100 * s.mean()),
    empty_map_pct=("empty", lambda s: 100 * s.mean()), sec_per_image=("sec", "mean"))
faith.to_csv(RES_DIR/"scorecam_faithfulness.csv")
display(faith.round(3))

## 17. Save config for the Streamlit app and a final summary

In [ ]:
app_cfg = dict(
    class_names=CLASS_NAMES, img_size=CFG["img_size"], mean=MEAN, std=STD,
    models={n: dict(checkpoint=f"models/{n}.pt", target_layer=TARGET_LAYER_EXPR[n], temperature=TEMP[n],
                    threshold=final_thr[n]) for n in MODEL_NAMES},
    ensemble=dict(members=MODEL_NAMES, threshold=final_thr["ensemble"]),
    operating_point=OPERATING_POINT,
    scorecam=dict(weight_mode=CAM_MODE, batch_size=64, default_target_class=TARGET_CLASS),
    test_metrics_with_ci=ci_table.to_dict(orient="index"),
    shortcut_check=json.loads(shortcut.to_json(orient="index")),
    faithfulness=faith.reset_index().to_dict(orient="records"),
)
(OUT_DIR/"app_config.json").write_text(json.dumps(app_cfg, indent=2, default=float))

print("Reportable test results (operating point chosen on validation, patient-level 95% CIs):")
display(ci_table)
print("\nSaved in", OUT_DIR)
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file(): print(" -", p.relative_to(OUT_DIR), f"({p.stat().st_size/1e6:.1f} MB)")

## How to describe these results honestly (README / CV)

- Report **AUC with its confidence interval** as the headline, plus sensitivity and specificity at the stated threshold. Say how the threshold was chosen (validation only).
- State the **test-set size** (624 images, 234 normal / 390 pneumonia) and that it comes from one public dataset collected at a single institution; no external validation was done.
- Mention the **limitations** you found: low specificity, shift between validation and test, DenseNet-121 attention outside the lungs (if the shortcut check and maps support it), and Score-CAM maps for NORMAL predictions.
- Do not claim clinical readiness, diagnosis, or "X% accuracy" without the context above.

In [ ]:
# Optional: zip everything and download to your computer
!cd /content/drive/MyDrive && zip -rq pneumonia_project.zip pneumonia_project
from google.colab import files
files.download("/content/drive/MyDrive/pneumonia_project.zip")